# Validation of ES-QC for the reference year

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import pandas as pd
from energyscope.models import Model
from energyscope.energyscope import Energyscope
from energyscope.result import postprocessing
from ast import literal_eval
from utils import *
import plotly.express as px
import plotly.graph_objects as go
import bw2data as bd
from tqdm import tqdm
from new_plots import _create_sankey_figure, generate_sankey_flows
from shared.utils import run_model, load_snapshot

In [ ]:
import plotly.io as pio
pio.renderers.default = "png"

In [ ]:
save_results = True
update_dat_files = False
reference_year = 2023

In [ ]:
if reference_year == 2021:
    N_capita = N_capita_2021 # observed number of people in Quebec in 2021 (https://statistique.quebec.ca/fr/document/projections-de-population-le-quebec/publication/le-quebec-projections-demographiques)
else:  # 2023
    N_capita = N_capita_2023

In [ ]:
# AMPL licence 
path_to_ampl_licence = r'C:\Users\matth\ampl' # Path to the AMPL license file
os.environ['PATH'] = path_to_ampl_licence+':'+os.environ['PATH']

In [ ]:
path_model = '../02_AMPL_files/model/'
path_data = f'../02_AMPL_files/data/{reference_year}/'
path_results = '../03_Results/LCA/'

In [ ]:
bd.projects.set_current('ecoinvent3.10.1')

In [ ]:
if update_dat_files:
    main_db = Database([
        f'ecoinvent_cutoff_3.10.1_image_SSP5-H_{reference_year}+truck_carculator',
        f'ei_3.10.1_image_SSP5-H_{reference_year}+truck_carculator_reg',
        f'ei_3.10.1_image_SSP5-H_{reference_year}+truck_carculator_reg_wo',
    ], create_pickle=True)

    update_ampl_files(year=2023, reg_level='all', specific_lcia_abbrev=['m_CCS_all'], main_database=main_db)

In [ ]:
reg_level = 'spat_fore_back' # can be 'base', 'spat', 'spat_back', 'spat_fore', 'spat_fore_back'

In [ ]:
path_lca_data = path_data+f'{reg_level}/'

In [ ]:
max_indicator = pd.read_csv(f'{path_lca_data}/QC_techs_lca_max.csv')
es_tech_df = pd.read_csv('../01_Notebooks/Data/technology_dictionary.csv')

In [ ]:
# Create a dict from the Programming Name and Long name columns of es_tech_dict
es_tech_name_dict = dict(zip(es_tech_df['Programming name'], es_tech_df['Long name']))
es_tech_name_dict = {k: v for k, v in es_tech_name_dict.items() if k not in wood_list+wet_biomass_list+waste_list}

## Initialize the model

In [ ]:
# Initialize the reference QC model with .mod and .dat files
model = load_snapshot(reference_year)

In [ ]:
# Add LCA files
model = model + Model([
    ('mod', path_model+'QC_objectives_lca.mod'),
    ('mod', path_model+'QC_objectives_lca_direct.mod'),
    ('mod', path_model+'QC_objectives_lca_territorial.mod'),
    ('dat', path_lca_data+'QC_techs_lca.dat'),
    ('dat', path_lca_data+'QC_techs_lca_direct.dat'),
    ('dat', path_lca_data+'QC_techs_lca_territorial.dat'),
])

In [ ]:
# Solve the model and get results
results = run_model(model)

In [ ]:
df_sankey = generate_sankey_flows(
        results=results,
        aggregate_mobility=True,
        aggregate_grid=True,
        aggregate_technology=True,
        run_id=0,
    )
df_sankey['source (long)'] = df_sankey.apply(lambda x: es_tech_name_dict[x['source']] if x['source'] in es_tech_name_dict else x['source'], axis=1)
df_sankey['target (long)'] = df_sankey.apply(lambda x: es_tech_name_dict[x['target']] if x['target'] in es_tech_name_dict else x['target'], axis=1)

fig = _create_sankey_figure(df_sankey, colors=default_colors_sankey, long_names=True)

if save_results:
    fig.write_html(f'../03_Results/Figures/reference/sankey_{reference_year}.html')

if save_results:
    df_sankey.to_csv(f'../03_Results/Tables/reference/sankey_raw_{reference_year}.csv', index=False)
df_sankey['value'] *= 1e-3 # from GWh to TWh

In [ ]:
# Imports, local and renewable resources
df_sankey_imports = df_sankey[df_sankey.source.str.startswith('IMP_')].groupby('source').sum()[['value']]
df_sankey_ren_res = df_sankey[(df_sankey.source.str.startswith('RES_')) | (df_sankey.source == 'WOOD')].groupby('source').sum()[['value']]

In [ ]:
es_end_use_sectors = ['HOUSEHOLDS', 'SERVICES', 'AGRICULTURE', 'INDUSTRY', 'MOBILITY_PASSENGER', 'MOBILITY_FREIGHT', 'EUD_ELECTRICITY_EHV_EXPORT']
es_end_use_types = {
    'EUD_ELECTRICITY_LV': 'ELECTRICITY',
    'EUD_ELECTRICITY_MV': 'ELECTRICITY',
    'EUD_ELECTRICITY_HV': 'ELECTRICITY',
    'EUD_ELECTRICITY_EHV': 'ELECTRICITY',
    'EUD_LIGHTING': 'ELECTRICITY',
    'ELECTRICITY': 'ELECTRICITY',
    'EUD_HEAT_LOW_T_SC': 'ELECTRICITY',
    'EUD_HEAT_LOW_T_SH': 'HEAT LOW T',
    'EUD_HEAT_LOW_T_HW': 'HEAT LOW T',
    'EUD_HEAT_HIGH_T': 'HEAT HIGH T',
}

def aggregate_end_use_types(row):
    if row['target'] in ['MOBILITY_FREIGHT', 'MOBILITY_PASSENGER']:
        return row['target']
    else:
        return es_end_use_types[row['source']]

df_sankey_demands = df_sankey[df_sankey.target.isin(es_end_use_sectors)][['target', 'source', 'value']]
df_sankey_demands['source'] = df_sankey_demands.apply(lambda x: aggregate_end_use_types(x), axis=1)
df_sankey_demands = df_sankey_demands.groupby(['source', 'target']).sum().reset_index().sort_values('target').set_index('target')

In [ ]:
df_sankey_imports['type'] = 'Imports'
df_sankey_ren_res['type'] = 'Production from renewable resources'
df_sankey_demands['type'] = 'Demands'

In [ ]:
df_sankey_summary = pd.concat([df_sankey_imports, df_sankey_ren_res, df_sankey_demands])
if save_results:
    df_sankey_summary.to_csv(f'../03_Results/Tables/reference/sankey_summary_{reference_year}.csv')

In [ ]:
es_eud = results.parameters['end_uses_demand_year'].reset_index().groupby('index0').sum()['end_uses_demand_year']
es_eud[es_eud.index.str.startswith('MOBILITY_FREIGHT')].sum()/1000, es_eud[es_eud.index.str.startswith('MOBILITY_PASSENGER')].sum()/1000

In [ ]:
if save_results:
    eud = results.parameters['end_uses_demand_year'].reset_index()
    eud[eud['end_uses_demand_year'] != 0].reset_index(drop=True).drop(columns=['Run']).to_csv(f'../03_Results/Tables/reference/eud_{reference_year}.csv', index=False)

## Validation

### Summary table

In [ ]:
model = pd.read_csv('../01_Notebooks/Data/model_2023.csv')

In [ ]:
# Annual production
annual_prod = results.variables['Annual_Prod']

In [ ]:
# Installed capacities 
installed_capacities = results.variables['F_Mult']

In [ ]:
# Annual resources
annual_resources = results.variables['Annual_Res']

In [ ]:
# Validation metrics extracted from StatCan and "État de l'énergie au Québec 2024" (data for 2021 and 2023)
if reference_year == 2021:
    validation_dict = {
        'WIND_ONSHORE': 11122, # onshore wind production [GWh]
        'HYDRO': 201250, # hydro production [GWh]
        'PV_ROOF': 14, # PV production [GWh]
        'ELECTRICITY_EHV': 32778, # electricity imports [GWh]
        'NG_EHP': 59867, # natural gas imports [GWh]
        'COAL': 3991, # coal and coke imports [GWh]
        'LFO': 28663, # oil imports for agriculture, industry and buildings [GWh]
        'HFO': 5594, # boat fuel [GWh]
        'DIESEL': 34046, # diesel imports [GWh]
        'GASOLINE': 69019, # gasoline imports [GWh]
        'DIRECT_GHG': 48120, # territorial GWP [kt CO2]
        'JETFUEL': 8470, # jet fuel imports [GWh]
        'WOOD': 36719, # wood production [GWh]
    }
else:
    validation_dict = {
        'WIND_ONSHORE': 12061, # onshore wind production [GWh]
        'HYDRO': 201500, # hydro production [GWh]
        'PV_ROOF': 14, # PV production [GWh]
        'ELECTRICITY_EHV': 33889, # electricity imports [GWh]
        'NG_EHP': 62180, # natural gas imports [GWh]
        'COAL': 3825, # coal and coke imports [GWh]
        'LFO': 26084, # oil imports for agriculture, industry and buildings [GWh]
        'HFO': 4885, # boat fuel [GWh]
        'DIESEL': 35142, # diesel imports [GWh]
        'GASOLINE': 74755, # gasoline imports [GWh]
        'DIRECT_GHG': 53450, # territorial GWP [kt CO2]
        'JETFUEL': 19896, # jet fuel imports [GWh]
        'WOOD': 43588, # wood production [GWh]
    }

In [ ]:
def compare(
        tech = None, 
        res = None, 
        emissions = None,
        n_digits = 3,
):
    if tech is not None:
        data_type = tech
        if tech == 'HYDRO':
            annual = annual_prod.loc['HYDRO_DAM'].Annual_Prod + annual_prod.loc['HYDRO_RIVER'].Annual_Prod # sum of hydro production from dams and rivers
        else:
            annual = annual_prod.loc[tech].Annual_Prod
    
    elif res is not None:
        data_type = res
        if res == 'WOOD':
            annual = annual_resources.loc[annual_resources.index.isin(wood_list)]['Annual_Res'].sum()
        else:
            annual = annual_resources.loc[res].Annual_Res
    
    elif emissions is not None:
        data_type = emissions
        # territorial_emissions = results.variables['TotalTERRITORIAL'].loc['m_CCS_all']['TotalTERRITORIAL']
        direct_emissions = results.variables['TotalDIRECT'].loc['m_CCS_all']['TotalDIRECT']
        annual = direct_emissions * max_indicator[max_indicator.Abbrev == 'm_CCS_all'].max_unit.iloc[0]
    
    else:
        raise ValueError("Provide either a technology or a resource to calculate the percentage difference")
    
    return [
        data_type, # technology or resource
        round(validation_dict[data_type]/1000, n_digits), # reference value in TWh or Mt CO2
        round(float(annual)/1000, n_digits), # model value in TWh or Mt CO2
        round(validation_dict[data_type]/1000 - float(annual)/1000, n_digits), # absolute difference in TWh or Mt CO2
        round((validation_dict[data_type] - float(annual)) / validation_dict[data_type], n_digits), # relative difference
    ]

In [ ]:
data = []
for res in ['NG_EHP', 'LFO', 'COAL', 'DIESEL', 'GASOLINE', 'JETFUEL', 'HFO', 'WOOD', 'ELECTRICITY_EHV']:
    data.append(compare(res=res))
# Technologies
for tech in ['HYDRO', 'WIND_ONSHORE', 'PV_ROOF']:
    data.append(compare(tech=tech))
# Emissions
for emissions in ['DIRECT_GHG']:
    data.append(compare(emissions=emissions))

df = pd.DataFrame(data, columns=['TECH', 'REF', 'ES', 'DELTA', 'RELATIVE_DELTA'])

In [ ]:
df_terr_res = results.variables['TERRITORIAL_res'].reset_index()
total_co2_uptake_wood = df_terr_res[df_terr_res['index1'].isin(wood_list)]['TERRITORIAL_res'].sum()
df_wood_uti = pd.merge(annual_prod[annual_prod.index.str.contains('WOOD')], model[model.Flow == 'WOOD'], how='left', left_index=True, right_on='Name')
df_wood_uti['WOOD'] = -1.0 * df_wood_uti['Annual_Prod'] * df_wood_uti['Amount']
df_wood_uti['CO2 uptake'] = total_co2_uptake_wood * df_wood_uti['WOOD'] / df_wood_uti['WOOD'].sum()

In [ ]:
df_op = pd.merge(
    results.variables['DIRECT_op'].reset_index(),
    results.postprocessing['df_annual'].reset_index()[['index', 'Category']],
    left_on='index1',
    right_on='index',
    how='left',
).rename(columns={'DIRECT_op': 'Value'}).drop(columns=['index0', 'index1', 'Run'])
df_op = df_op[df_op.Value != 0].reset_index(drop=True)
df_op['Sector'] = df_op.apply(category_to_sector, axis=1)
df_op['Type'] = 'Operation (direct)'
df_op = df_op.merge(df_wood_uti[['Name', 'CO2 uptake']], how='left', left_on='index', right_on='Name')
df_op['CO2 uptake'] = df_op['CO2 uptake'].fillna(0)
df_op['Value wo uptake'] = df_op['Value']  # keep track of previous values
df_op['Value'] += df_op['CO2 uptake']  # allocating the CO2 uptake of wood to the technologies for easier comparability with references

# df_constr = pd.merge(
#     results.variables['TERRITORIAL_constr'].reset_index(),
#     results.postprocessing['df_annual'].reset_index()[['index', 'Category']],
#     left_on='index1',
#     right_on='index',
#     how='left',
# ).rename(columns={'TERRITORIAL_constr': 'Value'}).drop(columns=['index0', 'index1', 'Run'])
# df_constr = df_constr[df_constr.Value != 0].reset_index(drop=True)
# df_constr['Sector'] = df_constr.apply(category_to_sector, axis=1)
# df_constr['Type'] = 'Construction'

df_res = results.variables['TERRITORIAL_res'].reset_index().rename(columns={'index1': 'index', 'TERRITORIAL_res': 'Value'}).drop(columns=['index0', 'Run'])
df_res = df_res[df_res.Value != 0].reset_index(drop=True)
df_res = df_res[~df_res['index'].isin(wood_list)]  # removing wood resources as their CO2 uptake is accounted in technologies
df_res['Sector'] = "Resources"
df_res['Type'] = 'Resources'

df_tot = pd.concat([df_op, df_res], ignore_index=True)

In [ ]:
total_lca = results.variables['TotalLCIA'].loc['m_CCS_all'].loc['TotalLCIA'] * max_indicator[max_indicator.Abbrev == 'm_CCS_all'].max_unit.iloc[0] / 1000 # Mt CO2

In [ ]:
df_sectors = df_tot[df_tot.Type != 'Construction'].groupby(['Sector']).sum()[['Value']] * max_indicator[max_indicator.Abbrev == 'm_CCS_all'].max_unit.iloc[0] / 1000 # Mt CO2
dict_sectors = dict(zip(df_sectors.index, df_sectors.Value))

In [ ]:
emissions_freight_plane = df_tot[(df_tot['index'] == 'PLANE_FREIGHT_ELD') & (df_tot['Type'] == 'Operation (direct)')]['Value'].sum() * max_indicator[max_indicator.Abbrev == 'm_CCS_all'].max_unit.iloc[0] / 1000 # Mt CO2
emissions_pass_plane = df_tot[(df_tot['index'] == 'PLANE_SH_LD') & (df_tot['Type'] == 'Operation (direct)')]['Value'].sum() * max_indicator[max_indicator.Abbrev == 'm_CCS_all'].max_unit.iloc[0] / 1000 # Mt CO2
emissions_maritime = df_tot[(df_tot['index'].isin(['BULK_CARRIER_ELD', 'CONTAINER_ELD', 'OIL_TANKER_ELD'])) & (df_tot['Type'] == 'Operation (direct)')]['Value'].sum() * max_indicator[max_indicator.Abbrev == 'm_CCS_all'].max_unit.iloc[0] / 1000 # Mt CO2
dict_sectors["Freight mobility"] -= (emissions_freight_plane + emissions_maritime)
dict_sectors["Passenger mobility"] -= emissions_pass_plane
dict_sectors["Plane"] = emissions_freight_plane + emissions_pass_plane
dict_sectors["Maritime freight"] = emissions_maritime
dict_sectors["Mobility"] = dict_sectors["Freight mobility"] + dict_sectors["Passenger mobility"]

In [ ]:
emissions_hydro = df_tot[(df_tot['index'] == 'HYDRO_DAM') & (df_tot['Type'] == 'Operation (direct)')]['Value'].sum() * max_indicator[max_indicator.Abbrev == 'm_CCS_all'].max_unit.iloc[0] / 1000 # Mt CO2
emissions_hydro_import = df_tot[(df_tot['index'] == 'ELECTRICITY_EHV') & (df_tot['Type'] == 'Resources')]['Value'].sum() * max_indicator[max_indicator.Abbrev == 'm_CCS_all'].max_unit.iloc[0] / 1000 # Mt CO2
emissions_heat_pumps = df_tot[(df_tot['index'] == 'DEC_HP_ELEC') & (df_tot['Type'] == 'Operation (direct)')]['Value'].sum() * max_indicator[max_indicator.Abbrev == 'm_CCS_all'].max_unit.iloc[0] / 1000 # Mt CO2
dict_sectors["Electricity"] -= (emissions_hydro + emissions_heat_pumps)
dict_sectors["Resources"] -= emissions_hydro_import
dict_sectors["Hydro"] = emissions_hydro + emissions_hydro_import
dict_sectors["HPs"] = emissions_heat_pumps
dict_sectors["Other"] = dict_sectors['Grid infrastructure'] + dict_sectors["Resources"]

In [ ]:
dict_sectors

In [ ]:
if reference_year == 2021:
    ghg_sectors_gouv = {
        "Mobility": 33.040,
        "Industrial heat": 11.6,
        "Domestic heat": 7.0,
        "Electricity": 0.38,
        "Hydro": 0,
        "HPs": 0,
        "Other": 0.23,
    }
else:
    ghg_sectors_gouv = {
        "Mobility": 34.89 - 0.8 - 0.91 - 1.09,  # remove air and maritime because data scope not aligned, removing off-road mobility (other) because not in ES
        "Maritime freight": 0.8,
        "Plane": 0.91,
        "Industrial heat": 11.7,
        "Domestic heat": 6.2,
        "Hydro": 0,
        "HPs": 0,
        "Other": 0.27+0.39,
    }

In [ ]:
if reference_year == 2021:
    ghg_sectors_ceudb = {
        "Industrial heat": 14.794,
        "Domestic heat": 3.038+4.142+0.339, # Residential + commercial + non-motive emissions from agriculture
        "Passenger mobility": 16.87,
        "Freight mobility": 13.47+1.381, # Normal freight + motive emissions from agriculture
        "Electricity": 0,
    }
else:
    ghg_sectors_ceudb = {
        "Industrial heat": 14.598,
        "Domestic heat": 2.905+4.002+0.323, # Residential + commercial + non-motive emissions from agriculture
        "Passenger mobility": 20.823,
        "Freight mobility": 15.692, # Normal freight + motive emissions from agriculture
        "Electricity": 0,
    }

In [ ]:
ghg_comp = []
for key in ghg_sectors_gouv.keys():
    ghg_comp.append([
        f"DIRECT_GHG - {key}",
        round(ghg_sectors_gouv[key], 3),
        round(dict_sectors[key], 3),
        round(ghg_sectors_gouv[key] - dict_sectors[key], 3),
        round((ghg_sectors_gouv[key] - dict_sectors[key]) / ghg_sectors_gouv[key], 3) if ghg_sectors_gouv[key] != 0 else None,
    ])

In [ ]:
df = pd.concat([df, pd.DataFrame(ghg_comp, columns=['TECH', 'REF', 'ES', 'DELTA', 'RELATIVE_DELTA'])])

In [ ]:
if save_results:
    df.to_csv(f'../03_Results/Tables/reference/validation_table_{reference_year}.csv', index=False)

In [ ]:
df

In [ ]:
for length in ['_SD', '_MD', '_LD', '_ELD']:
    df_tot['index'] = df_tot['index'].str.replace(length, '')

In [ ]:
df_verif = df_tot.groupby(['index', 'Sector', 'Type'])['Value'].sum() * max_indicator[max_indicator.Abbrev == 'm_CCS_all'].max_unit.iloc[0] / 1000 # Mt CO2
if save_results:
    df_verif.reset_index().sort_values('Sector').to_csv('../03_Results/Tables/reference/verif_direct_emissions.csv')

## Impact of the regionalization level on LCA results

In [ ]:
biosphere_db = Database('biosphere3')
spatialized_biosphere = Database("biosphere3_spatialized_flows")
full_biosphere_db = biosphere_db + spatialized_biosphere
full_biosphere_db_as_dict_code = full_biosphere_db.list_to_dict(key='code', database_type='biosphere')

In [ ]:
impact_abbrev = pd.read_csv('../01_Notebooks/Data/impact_abbrev.csv')

### Areas of protection

In [ ]:
aop_impact_categories_list = [
    ('IMPACT World+ Damage 2.1_regionalized for ecoinvent v3.10', 'Ecosystem quality', 'Total ecosystem quality (biogenic)'),
    ('IMPACT World+ Damage 2.1_regionalized for ecoinvent v3.10', 'Human health', 'Total human health (biogenic)'),
]

In [ ]:
endpoint_impact_categories_list = [
    i for i in bd.methods if
    (i[0] == 'IMPACT World+ Damage 2.1_regionalized for ecoinvent v3.10')
    | (i[0] == 'IMPACT World+ Damage 2.1 for ecoinvent v3.10 (incl. CO2 uptake)')
]

endpoint_impact_categories_list +=[
    ('IMPACT World+ Damage 2.1_regionalized for ecoinvent v3.10', 'Ecosystem quality', 'Remaining ecosystem quality'),
    ('IMPACT World+ Damage 2.1_regionalized for ecoinvent v3.10', 'Human health', 'Remaining human health'),
    ('IMPACT World+ Damage 2.1_regionalized for ecoinvent v3.10', 'Ecosystem quality', 'Total ecosystem quality (biogenic)'),
    ('IMPACT World+ Damage 2.1_regionalized for ecoinvent v3.10', 'Human health', 'Total human health (biogenic)'),
]

endpoint_impact_categories_list = [i for i in endpoint_impact_categories_list if i not in [  # keep only marine acidification from -1/+1 version of IW+
    ('IMPACT World+ Damage 2.1_regionalized for ecoinvent v3.10', 'Ecosystem quality', 'Marine acidification, short term'),
    ('IMPACT World+ Damage 2.1_regionalized for ecoinvent v3.10', 'Ecosystem quality', 'Marine acidification, long term')
]]

In [ ]:
regionalized_endpoint_impact_categories = []
not_regionalized_endpoint_impact_categories = []
for i in endpoint_impact_categories_list:
    if 'Total' in i[-1] or 'Remaining' in i[-1]:
        pass
    elif i[0] == 'IMPACT World+ Damage 2.1 for ecoinvent v3.10 (incl. CO2 uptake)':
        not_regionalized_endpoint_impact_categories.append(i)
    else:
        if impact_abbrev[impact_abbrev['Impact_category'] == str(i)]['Regionalized'].iloc[0] == True:
            regionalized_endpoint_impact_categories.append(i)
        else:
            not_regionalized_endpoint_impact_categories.append(i)

In [ ]:
results_phases_list = []
results_phases_list_cc = []
results_categories_list = []

list_df_f_mult = []
list_df_annual_prod = []
list_df_annual_res = []

list_df_contrib_analysis_ef_constr = []
list_df_contrib_analysis_ef_op = []
list_df_contrib_analysis_ef_res = []

impact_category = 'Human health (biogenic)' # 'Human health (biogenic)' or 'Ecosystem quality (biogenic)'
contribution_ef = False

for reg_level in tqdm(['base', 'spat', 'spat_back', 'spat_fore', 'spat_fore_back']):

    # Loading LCA results
    impact_scores = pd.read_csv(path_results+f'2023/{reg_level}/impact_scores.csv')
    impact_scores_direct = pd.read_csv(path_results+f'2023/{reg_level}/impact_scores_direct_emissions.csv')
    max_indicator = pd.read_csv(f'../02_AMPL_files/data/2023/{reg_level}/QC_techs_lca_max.csv')
    max_ccs_norm = max_indicator[max_indicator.Abbrev == 'm_CCS_all'].max_unit.iloc[0]

    impact_scores, impact_abbrev = add_biogenic_climate_change_to_impact_scores_df(impact_scores, impact_abbrev)
    impact_scores_direct = add_biogenic_climate_change_to_impact_scores_df(impact_scores_direct, impact_abbrev)[0]

    impact_scores, impact_abbrev = add_rhhd_and_reqd_to_impact_scores_df(impact_scores, impact_abbrev)
    impact_scores_direct = add_rhhd_and_reqd_to_impact_scores_df(impact_scores_direct, impact_abbrev)[0]

    # from [impact / kW(h) or pkm(/h) or tkm(/h)] to [impact / GW(h) or Mpkm(/h) or Mtkm(/h)]
    impact_scores.Value *= 1e6
    impact_scores_direct.Value *= 1e6

    # Reading impact categories as tuples
    impact_scores.Impact_category = impact_scores.Impact_category.apply(lambda x: literal_eval(x))
    impact_scores_direct.Impact_category = impact_scores_direct.Impact_category.apply(lambda x: literal_eval(x))

    endpoint_aop_impact_categories_list = [i for i in endpoint_impact_categories_list if i[1] == impact_category.replace(' (biogenic)', '')]

    results = run_opti(
        reg_level=reg_level,
        validation=True,
        year=2023,
        other_emissions=True,
        constraint_on_remaining_aop=False,
        constraint_on_foreign_ghg_emissions=False,
        constraint_on_territorial_ghg_emissions=False,
    )

    all_mob_techs = (
        list(results.sets['TECHNOLOGIES_OF_FREIGHTMOB_ALL_DISTANCES']['TECHNOLOGIES_OF_FREIGHTMOB_ALL_DISTANCES'])
        + list(results.sets['TECHNOLOGIES_OF_PRIVATEMOB_ALL_DISTANCES']['TECHNOLOGIES_OF_PRIVATEMOB_ALL_DISTANCES'])
        + list(results.sets['TECHNOLOGIES_OF_PUBLICMOB_ALL_DISTANCES']['TECHNOLOGIES_OF_PUBLICMOB_ALL_DISTANCES'])
    )

    # Merging impact scores with energy configuration results
    df_f_mult, df_annual_prod, df_annual_res = get_impact_scores(
        impact_category=endpoint_impact_categories_list+[('IMPACT World+ Midpoint 2.1 for ecoinvent v3.10 (incl. CO2 uptake)', 'Midpoint', 'Climate change, short term, total')],
        df_impact_scores=impact_scores,
        df_results=results,
    )

    df_annual_prod_direct = get_impact_scores(
        impact_category=endpoint_impact_categories_list+[('IMPACT World+ Midpoint 2.1 for ecoinvent v3.10 (incl. CO2 uptake)', 'Midpoint', 'Climate change, short term, total')],
        df_impact_scores=impact_scores_direct,
        df_results=results,
        assessment_type='direct',
    )

    df_annual_prod = df_annual_prod.merge(
        results.variables['ABROAD_op'].loc['m_CCS_all'].drop(columns=['Run']) * max_ccs_norm, left_on='index', right_index=True,
    ).rename(columns={'ABROAD_op': 'Climate change, short term, total (abroad)'})
    df_annual_prod = df_annual_prod.merge(
        results.variables['TERRITORIAL_op'].loc['m_CCS_all'].drop(columns=['Run']) * max_ccs_norm, left_on='index', right_index=True,
    ).rename(columns={'TERRITORIAL_op': 'Climate change, short term, total (territorial)'})
    df_f_mult = df_f_mult.merge(
        results.variables['ABROAD_constr'].loc['m_CCS_all'].drop(columns=['Run']) * max_ccs_norm, left_on='index', right_index=True,
    ).rename(columns={'ABROAD_constr': 'Climate change, short term, total (abroad)'})
    df_f_mult = df_f_mult.merge(
        results.variables['TERRITORIAL_constr'].loc['m_CCS_all'].drop(columns=['Run']) * max_ccs_norm, left_on='index', right_index=True,
    ).rename(columns={'TERRITORIAL_constr': 'Climate change, short term, total (territorial)'})
    df_annual_res = df_annual_res.merge(
        results.variables['ABROAD_res'].loc['m_CCS_all'].drop(columns=['Run']) * max_ccs_norm, left_on='index', right_index=True,
    ).rename(columns={'ABROAD_res': 'Climate change, short term, total (abroad)'})
    df_annual_res = df_annual_res.merge(
        results.variables['TERRITORIAL_res'].loc['m_CCS_all'].drop(columns=['Run']) * max_ccs_norm, left_on='index', right_index=True,
    ).rename(columns={'TERRITORIAL_res': 'Climate change, short term, total (territorial)'})

    df_f_mult['Run'] = reg_level
    df_annual_prod['Run'] = reg_level
    df_annual_prod_direct['Run'] = reg_level
    df_annual_res['Run'] = reg_level

    # Keeping mobility sub-models only
    df_f_mult = df_f_mult[~df_f_mult['index'].isin(all_mob_techs)]
    df_annual_prod = df_annual_prod[~df_annual_prod['index'].isin(all_mob_techs)]
    df_annual_prod_direct = df_annual_prod_direct[~df_annual_prod_direct['index'].isin(all_mob_techs)]

    df_annual_prod_direct.columns = [i + ' (direct)' if i not in ['index', 'Annual_Prod', 'Category', 'Run'] else i for i in df_annual_prod_direct.columns]

    df_annual_prod = df_annual_prod.merge(df_annual_prod_direct, on=['index', 'Annual_Prod', 'Category', 'Run'])

    for cat in [f'Total {impact_category.lower()}', 'Climate change, short term, total']:
        impact_constr = df_f_mult[cat].sum() / N_capita
        impact_op = df_annual_prod[cat].sum() / N_capita
        impact_op_direct = df_annual_prod_direct[f'{cat} (direct)'].sum() / N_capita
        impact_res_wo_biomass = df_annual_res[~df_annual_res['index'].isin(wood_list+wet_biomass_list+waste_list)][cat].sum() / N_capita
        impact_res_biomass = df_annual_res[df_annual_res['index'].isin(wood_list+wet_biomass_list+waste_list)][cat].sum() / N_capita
        if cat == f'Total {impact_category.lower()}':
            results_phases_list.append([
                reg_level_name_dict[reg_level],
                impact_constr,
                impact_op,
                impact_op_direct,
                impact_res_wo_biomass,
                impact_res_biomass,
            ])
        else:
            impact_constr *= 1e-3
            impact_op *= 1e-3
            impact_op_direct *= 1e-3
            impact_res_wo_biomass *= 1e-3
            impact_res_biomass *= 1e-3
            results_phases_list_cc.append([
                reg_level_name_dict[reg_level],
                impact_constr,
                impact_op,
                impact_op_direct,
                impact_res_wo_biomass,
                impact_res_biomass,
            ])

    for i in endpoint_aop_impact_categories_list:
        cat = i[2]
        if cat not in [f'Total {impact_category.lower()}', f'Remaining {impact_category.replace(" (biogenic)", "").lower()}']:
            impact = (df_f_mult[cat].sum() + df_annual_prod[cat].sum() + df_annual_res[cat].sum()) / N_capita
            impact_direct = df_annual_prod_direct[cat + ' (direct)'].sum() / N_capita
            results_categories_list.append([
                reg_level_name_dict[reg_level],
                cat,
                i in regionalized_endpoint_impact_categories,
                impact,
                impact_direct,
            ])

    # Contribution analysis
    if contribution_ef:

        if reg_level != 'base': # skip base level for EF contributions (impact location)

            results_contrib_ef = pd.read_csv(f'../03_Results/LCA/2020/{reg_level}/contribution_analysis_emissions.csv')
            results_contrib_ef.impact_category = results_contrib_ef.impact_category.apply(lambda x: literal_eval(x))
            results_contrib_ef = results_contrib_ef[results_contrib_ef.impact_category.isin(regionalized_endpoint_impact_categories)]
            results_contrib_ef.score *= 1e6
            results_contrib_ef.amount *= 1e6
            results_contrib_ef['name'] = results_contrib_ef.apply(lambda x: full_biosphere_db_as_dict_code[(x['database'], x['code'])]['name'], axis=1)

            results_contrib_ef_constr = results_contrib_ef[results_contrib_ef.act_type == 'Construction']
            results_contrib_ef_op = results_contrib_ef[results_contrib_ef.act_type == 'Operation']
            results_contrib_ef_res = results_contrib_ef[results_contrib_ef.act_type == 'Resource']

            results_contrib_ef_constr = results_contrib_ef_constr.merge(
                df_f_mult[['index', 'F_Mult', 'Run', 'lifetime']],
                left_on=['act_name'],
                right_on=['index'],
                how='left',
            )

            results_contrib_ef_op = results_contrib_ef_op.merge(
                df_annual_prod[['index', 'Annual_Prod', 'Run']],
                left_on=['act_name'],
                right_on=['index'],
                how='left',
            )

            results_contrib_ef_res = results_contrib_ef_res.merge(
                df_annual_res[['index', 'Annual_Res', 'Run']],
                left_on=['act_name'],
                right_on=['index'],
                how='left',
            )

            results_contrib_ef_constr['scaled_score'] = results_contrib_ef_constr['score'] * results_contrib_ef_constr['F_Mult'] / (results_contrib_ef_constr['lifetime'] * N_capita)
            results_contrib_ef_op['scaled_score'] = results_contrib_ef_op['score'] * results_contrib_ef_op['Annual_Prod'] / N_capita
            results_contrib_ef_res['scaled_score'] = results_contrib_ef_res['score'] * results_contrib_ef_res['Annual_Res'] / N_capita

            results_contrib_ef_constr['scaled_amount'] = results_contrib_ef_constr['amount'] * results_contrib_ef_constr['F_Mult'] / (results_contrib_ef_constr['lifetime'] * N_capita)
            results_contrib_ef_op['scaled_amount'] = results_contrib_ef_op['amount'] * results_contrib_ef_op['Annual_Prod'] / N_capita
            results_contrib_ef_res['scaled_amount'] = results_contrib_ef_res['amount'] * results_contrib_ef_res['Annual_Res'] / N_capita

            results_contrib_ef_constr = results_contrib_ef_constr[results_contrib_ef_constr['scaled_score'] != 0]
            results_contrib_ef_op = results_contrib_ef_op[results_contrib_ef_op['scaled_score'] != 0]
            results_contrib_ef_res = results_contrib_ef_res[results_contrib_ef_res['scaled_score'] != 0]

            results_contrib_ef_constr['location'] = results_contrib_ef_constr.apply(
                lambda x: get_ef_location(x['name'], x['database']), axis=1
            )
            results_contrib_ef_op['location'] = results_contrib_ef_op.apply(
                lambda x: get_ef_location(x['name'], x['database']), axis=1
            )
            results_contrib_ef_res['location'] = results_contrib_ef_res.apply(
                lambda x: get_ef_location(x['name'], x['database']), axis=1
            )

            list_df_contrib_analysis_ef_constr.append(results_contrib_ef_constr)
            list_df_contrib_analysis_ef_op.append(results_contrib_ef_op)
            list_df_contrib_analysis_ef_res.append(results_contrib_ef_res)

    df_f_mult = df_f_mult[df_f_mult.F_Mult != 0]
    df_annual_prod = df_annual_prod[df_annual_prod.Annual_Prod != 0]
    df_annual_res = df_annual_res[df_annual_res.Annual_Res != 0]

    list_df_f_mult.append(df_f_mult)
    list_df_annual_prod.append(df_annual_prod)
    list_df_annual_res.append(df_annual_res)

df_f_mult = pd.concat(list_df_f_mult)
df_annual_prod = pd.concat(list_df_annual_prod)
df_annual_res = pd.concat(list_df_annual_res)

df_f_mult = df_f_mult.merge(model[model['Amount'] == 1], left_on='index', right_on='Name', how='left').rename(columns={'Flow': 'Main production'}).drop(columns=['Name', 'Amount'])
df_annual_prod = df_annual_prod.merge(model[model['Amount'] == 1], left_on='index', right_on='Name', how='left').rename(columns={'Flow': 'Main production'}).drop(columns=['Name', 'Amount'])
df_f_mult['Main production'] = df_f_mult['Main production'].astype(str)
df_annual_prod['Main production'] = df_annual_prod['Main production'].astype(str)

df_annual_prod['Sector'] = df_annual_prod.apply(category_to_sector, axis=1)
df_f_mult['Sector'] = df_f_mult.apply(category_to_sector, axis=1)
df_annual_res['Sector'] = df_annual_res.apply(lambda x: 'Electricity' if x['index'] == 'ELECTRICITY_EHV' else ('Biomass' if x['index'] in wood_list+wet_biomass_list+waste_list else 'Imports'), axis=1)
df_annual_res['Category'] = df_annual_res.apply(lambda x: 'ELECTRICITY_EHV' if x['index'] == 'ELECTRICITY_EHV' else ('Biomass' if x['index'] in wood_list+wet_biomass_list+waste_list else 'Imports'), axis=1)

if contribution_ef:
    df_contrib_analysis_ef_constr = pd.concat(list_df_contrib_analysis_ef_constr)
    df_contrib_analysis_ef_op = pd.concat(list_df_contrib_analysis_ef_op)
    df_contrib_analysis_ef_res = pd.concat(list_df_contrib_analysis_ef_res)

if save_results:
    df_f_mult.to_csv('../03_Results/Tables/reference/df_f_mult.csv', index=False)
    df_annual_prod.to_csv('../03_Results/Tables/reference/df_annual_prod.csv', index=False)
    df_annual_res.to_csv('../03_Results/Tables/reference/df_annual_res.csv', index=False)

    if contribution_ef:
        df_contrib_analysis_ef_constr.to_csv('../03_Results/Tables/reference/df_contrib_analysis_ef_constr.csv', index=False)
        df_contrib_analysis_ef_op.to_csv('../03_Results/Tables/reference/df_contrib_analysis_ef_op.csv', index=False)
        df_contrib_analysis_ef_res.to_csv('../03_Results/Tables/reference/df_contrib_analysis_ef_res.csv', index=False)

In [ ]:
# To skip the previous cell
# df_f_mult = pd.read_csv('../03_Results/Tables/reference/df_f_mult.csv')
# df_annual_prod = pd.read_csv('../03_Results/Tables/reference/df_annual_prod.csv')
# df_annual_res = pd.read_csv('../03_Results/Tables/reference/df_annual_res.csv')
#
# if contribution_ef:
#     df_contrib_analysis_ef_constr = pd.read_csv('../03_Results/Tables/reference/df_contrib_analysis_ef_constr.csv')
#     df_contrib_analysis_ef_op = pd.read_csv('../03_Results/Tables/reference/df_contrib_analysis_ef_op.csv')
#     df_contrib_analysis_ef_res = pd.read_csv('../03_Results/Tables/reference/df_contrib_analysis_ef_res.csv')

In [ ]:
# Where does the difference in PMF come from (for human health damage analysis)
if 'Human health' in impact_category:
    df_pmf = df_annual_prod[['index', 'Run', 'Particulate matter formation']].dropna()
    df_pmf = df_pmf[(df_pmf['Run'] == 'spat') | (df_pmf['Run'] == 'spat_fore')]
    df_pmf_pivot = df_pmf.pivot_table(
        index='index',
        columns='Run',
        values='Particulate matter formation'
    ).reset_index().rename(columns={'spat': 'spat PMF', 'spat_fore': 'spat_fore PMF'})

    df_pmf_pivot['PMF difference'] = df_pmf_pivot['spat PMF'] - df_pmf_pivot['spat_fore PMF']
    df_pmf_pivot['PMF difference ratio'] = df_pmf_pivot['PMF difference'] / df_pmf_pivot['PMF difference'].sum()

In [ ]:
df_results_phases = pd.DataFrame(
    results_phases_list,
    columns=['Regionalization level', 'Construction', 'Operation', 'Operation (direct)', 'Resource (wo biomass)', 'Resources (biomass)']
)

In [ ]:
df_results_phases["Operation (indirect)"] = df_results_phases["Operation"] - df_results_phases["Operation (direct)"]
df_results_phases.drop(columns=["Operation"], inplace=True)

In [ ]:
df_results_phases = df_results_phases.melt(
    id_vars=['Regionalization level'],
    var_name='Life-cycle phase',
    value_name=impact_category,
)

In [ ]:
df_results_categories = pd.DataFrame(
    results_categories_list, 
    columns=['Regionalization level', 'Impact category', 'Regionalized', "Total", "Direct"],
)

In [ ]:
df_results_categories["Indirect"] = df_results_categories["Total"] - df_results_categories["Direct"]

In [ ]:
df_results_categories_grouped = df_results_categories.groupby(['Regionalization level', 'Impact category']).sum().reset_index()[['Regionalization level', 'Impact category', "Total"]]
df_results_categories_grouped['% variation wrt def'] = df_results_categories_grouped.apply(lambda x: 100 * (df_results_categories_grouped[(df_results_categories_grouped['Regionalization level'] == 'Def.') & (df_results_categories_grouped['Impact category'] == x['Impact category'])]['Total'].iloc[0] - x['Total']) / df_results_categories_grouped[(df_results_categories_grouped['Regionalization level'] == 'Def.') & (df_results_categories_grouped['Impact category'] == x['Impact category'])]['Total'].iloc[0], axis=1)

In [ ]:
df_results_categories_grouped = df_results_categories.groupby(['Regionalization level']).sum().reset_index()[['Regionalization level', "Direct", "Total"]]
df_results_categories_grouped['Direct (%)'] = 100 * df_results_categories_grouped['Direct'] / df_results_categories_grouped['Total']
total_ref = df_results_categories_grouped[df_results_categories_grouped['Regionalization level'] == 'Def.']['Total'].iloc[0]
df_results_categories_grouped['% variation wrt def'] = 100 * (df_results_categories_grouped['Total'] - total_ref) / total_ref
df_results_categories_grouped

In [ ]:
100 * (df_results_categories_grouped[df_results_categories_grouped['Regionalization level'] == 'Spat.']['Total'].iloc[0] - df_results_categories_grouped[df_results_categories_grouped['Regionalization level'] == 'Spat.+Back.']['Total'].iloc[0]) / df_results_categories_grouped[df_results_categories_grouped['Regionalization level'] == 'Spat.']['Total'].iloc[0]

In [ ]:
100 * (df_results_categories_grouped[df_results_categories_grouped['Regionalization level'] == 'Spat.+Fore.']['Total'].iloc[0] - df_results_categories_grouped[df_results_categories_grouped['Regionalization level'] == 'Spat.+Fore.+Back.']['Total'].iloc[0]) / df_results_categories_grouped[df_results_categories_grouped['Regionalization level'] == 'Spat.+Fore.']['Total'].iloc[0]

In [ ]:
df_results_categories = df_results_categories.melt(
    id_vars=['Regionalization level', 'Impact category', 'Regionalized'],
    value_vars=["Direct", "Indirect"],
    var_name='Scope',
    value_name=impact_category,
)

#### Contribution per impact category + direct vs indirect emissions

In [ ]:
if 'Human health' in impact_category:
    cutoff = 0.02
elif 'Ecosystem quality' in impact_category:
    cutoff = 0.03
else:
    raise ValueError("Impact category must be either 'Human health' or 'Ecosystem quality'")

In [ ]:
df_results_categories = df_results_categories[~df_results_categories['Impact category'].isin([
    f'Total {impact_category.replace(" (biogenic)", "").lower()}',
    f'Total {impact_category.lower()}',
    f'Remaining {impact_category.replace(" (biogenic)", "").lower()}',
    f'Climate change, {impact_category.replace(" (biogenic)", "").lower()}, short term',
    f'Climate change, {impact_category.replace(" (biogenic)", "").lower()}, short term, total',
    f'Climate change, {impact_category.replace(" (biogenic)", "").lower()}, long term',
    f'Climate change, {impact_category.replace(" (biogenic)", "").lower()}, long term, total'
])]

In [ ]:
plot_impact_categories_contribution(
    df_results_categories=df_results_categories,
    impact_category=impact_category,
    save_results=save_results,
    cutoff=cutoff,
    show_direct_emissions_markers=False,
    separate_negative_bars=True,
)

In [ ]:
df_cc_vs_tot = pd.merge(
    df_results_categories[df_results_categories['Impact category'].str.contains('Climate change')].groupby(['Regionalization level']).sum()[[impact_category]].rename(columns={impact_category: 'CC'}),
    df_results_categories.groupby(['Regionalization level']).sum()[[impact_category]].rename(columns={impact_category: 'Total'}),
    left_index=True,
    right_index=True,
)
df_cc_vs_tot['CC (%)'] = 100 * df_cc_vs_tot['CC'] / df_cc_vs_tot['Total']
df_cc_vs_tot

In [ ]:
df_direct_cc_vs_tot = pd.merge(
    df_results_categories[(df_results_categories['Impact category'].str.contains('Climate change')) & (df_results_categories['Scope'] == 'Direct')].groupby(['Regionalization level']).sum()[[impact_category]].rename(columns={impact_category: 'Direct CC'}),
    df_results_categories.groupby(['Regionalization level']).sum()[[impact_category]].rename(columns={impact_category: 'Total'}),
    left_index=True,
    right_index=True,
)
df_direct_cc_vs_tot['Direct CC (%)'] = 100 * df_direct_cc_vs_tot['Direct CC'] / df_direct_cc_vs_tot['Total']
df_direct_cc_vs_tot

In [ ]:
if save_results:
    df_results_categories.to_csv(f'../03_Results/Tables/reference/df_contrib_imp_cat_{"tthh" if impact_category == "Human health (biogenic)" else "tteq"}.csv', index=False)

#### Contribution of technologies/sectors

In [ ]:
df_annual_prod = aggregate_mobility_submodels(df_annual_prod)
df_f_mult = aggregate_mobility_submodels(df_f_mult)

In [ ]:
df_annual_prod['Run'] = df_annual_prod['Run'].replace(reg_level_name_dict)
df_f_mult['Run'] = df_f_mult['Run'].replace(reg_level_name_dict)
df_annual_res['Run'] = df_annual_res['Run'].replace(reg_level_name_dict)

df_annual_prod['index'] = df_annual_prod['index'].replace(es_tech_name_dict)
df_f_mult['index'] = df_f_mult['index'].replace(es_tech_name_dict)
df_annual_res['index'] = df_annual_res['index'].replace(es_tech_name_dict)

In [ ]:
for col in list(df_annual_prod.columns):
    if col not in ['index', 'Run', 'Sector', 'Category', 'Annual_Prod', 'Phase', 'Main production', 'Climate change, short term, total (territorial)', 'Climate change, short term, total (abroad)'] and "(direct)" not in col:
        df_annual_prod[f"{col} (indirect)"] = df_annual_prod[col] - df_annual_prod[f"{col} (direct)"]

In [ ]:
df_annual_prod_direct = df_annual_prod.copy(deep=True)
df_annual_prod_indirect = df_annual_prod.copy(deep=True)
for col in list(df_annual_prod.columns):
    if col not in ['index', 'Run', 'Sector', 'Category', 'Annual_Prod', 'Phase']:
        if "(territorial)" in col or "(abroad)" in col:
            df_annual_prod_direct[col] = 0  # phases are won't be used for these two categories
        elif "(direct)" in col:
            col = col.replace(" (direct)", "")
            df_annual_prod_direct = df_annual_prod_direct.rename(columns={f"{col} (direct)": col})
            df_annual_prod_indirect = df_annual_prod_indirect.drop(columns=[f"{col} (direct)"])
        elif "(indirect)" in col:
            col = col.replace(" (indirect)", "")
            df_annual_prod_direct = df_annual_prod_direct.drop(columns=[f"{col} (indirect)"])
            df_annual_prod_indirect = df_annual_prod_indirect.rename(columns={f"{col} (indirect)": col})
        else:
            df_annual_prod_direct = df_annual_prod_direct.drop(columns=[col])
            df_annual_prod_indirect = df_annual_prod_indirect.drop(columns=[col])

In [ ]:
col = ['Run', 'index', 'Sector', 'Phase', f'Total {impact_category.lower()}', f'Remaining {impact_category.replace(" (biogenic)", "").lower()}', 'Climate change, short term, total', 'Climate change, short term, total (abroad)', 'Climate change, short term, total (territorial)']
df_f_mult['Phase'] = 'Construction'
df_annual_prod['Phase'] = 'Operation'
df_annual_prod_direct['Phase'] = 'Operation (direct)'
df_annual_prod_indirect['Phase'] = 'Operation (indirect)'
df_annual_res['Phase'] = 'Resource'
df_total_impact = pd.concat([
    df_f_mult[['F_Mult'] + col].rename(columns={'F_Mult': 'Capacity or production'}),
    df_annual_prod_direct[['Annual_Prod'] + col].rename(columns={'Annual_Prod': 'Capacity or production'}),
    df_annual_prod_indirect[['Annual_Prod'] + col].rename(columns={'Annual_Prod': 'Capacity or production'}),
    df_annual_res[['Annual_Res'] + col].rename(columns={'Annual_Res': 'Capacity or production'}),
],
    ignore_index=True)

In [ ]:
df_total_impact[f'Total {impact_category.lower()}'] *= 1/N_capita
df_total_impact[f'Remaining {impact_category.replace(" (biogenic)", "").lower()}'] *= 1/N_capita
df_total_impact['Climate change, short term, total'] *= 1/(1e3 * N_capita)

In [ ]:
df_total_impact['Climate change, short term, total (abroad)'] *= 1e3/N_capita
df_total_impact['Climate change, short term, total (territorial)'] *= 1e3/N_capita

In [ ]:
if save_results:
    df_config = df_total_impact[(df_total_impact.Run == 'Def.') & (df_total_impact.Phase != 'Operation (indirect)')][['index', 'Sector', 'Phase', 'Capacity or production']].rename(columns={'index': 'Technology or resource'})
    df_config.Phase = df_config.Phase.str.replace(' (direct)', '')
    df_config.to_csv(f'../03_Results/Tables/reference/df_config.csv', index=False)
    df_total_impact_saved = (df_total_impact.melt(value_vars=[
        f'Total {impact_category.lower()}',
        f'Remaining {impact_category.replace(" (biogenic)", "").lower()}',
        'Climate change, short term, total',
        'Climate change, short term, total (abroad)',
        'Climate change, short term, total (territorial)'
    ], id_vars=['Run', 'index', 'Sector', 'Phase']
    ).rename(columns={'index': 'Technology or resource', 'variable': 'Impact category', 'value': 'Value'}))
    df_total_impact_saved['Impact category'] = df_total_impact_saved['Impact category'].apply(lambda x: x.replace(' (biogenic)', ''))
    df_total_impact_saved['Impact category'] = df_total_impact_saved['Impact category'].apply(lambda x: x.replace(', total', ''))
    df_total_impact_saved.to_csv(f'../03_Results/Tables/reference/df_contrib_sector_tec_{"tthh" if impact_category == "Human health (biogenic)" else "tteq"}.csv', index=False)

In [ ]:
plot_contribution_by_sector(
    df=df_total_impact.rename(columns={f'Total {impact_category.lower()}': f'Total {impact_category.replace(" (biogenic)", "").lower()}'}),
    imp_cat=f'Total {impact_category.replace(" (biogenic)", "").lower()}',
    save_results=save_results,
    hatch_phase=False,
    showlegend=True,
    year=2020,
    show_regionalized_marker=False,
    show_total_marker=True,
    show_direct_marker=False,
    x_label_name="Regionalization level",
)

In [ ]:
plot_contribution_by_sector(
    df=df_total_impact.rename(columns={f'Total {impact_category.lower()}': f'Total {impact_category.replace(" (biogenic)", "").lower()}'}),
    imp_cat=f'Total {impact_category.replace(" (biogenic)", "").lower()}',
    save_results=save_results,
    group_by='index',
    cutoff=0.03,
    showlegend=True,
    year=2020,
    show_regionalized_marker=False,
    show_direct_marker=False,
    show_total_marker=True,
    x_label_name="Regionalization level",
)

In [ ]:
plot_contribution_by_sector(
    df=df_total_impact,
    imp_cat=f'Remaining {impact_category.replace(" (biogenic)", "").lower()}',
    save_results=save_results,
    hatch_phase=False,
    showlegend=True,
    year=2020,
    show_regionalized_marker=False,
    show_total_marker=True,
    show_direct_marker=False,
    x_label_name="Regionalization level",
)

In [ ]:
plot_contribution_by_sector(
    df=df_total_impact,
    imp_cat=f'Remaining {impact_category.replace(" (biogenic)", "").lower()}',
    save_results=save_results,
    group_by='index',
    cutoff=0.03,
    hatch_phase=False,
    showlegend=True,
    year=2020,
    show_regionalized_marker=False,
    show_total_marker=True,
    show_direct_marker=False,
    x_label_name="Regionalization level",
)

#### Regionalized impacts location

In [ ]:
if contribution_ef:
    # Filtering only for total AoP category
    df_contrib_analysis_ef_constr_aop = df_contrib_analysis_ef_constr[
        df_contrib_analysis_ef_constr.impact_category == ('IMPACT World+ Damage 2.1_regionalized for ecoinvent v3.10', impact_category, f'Total {impact_category.lower()} (biogenic)')
    ].groupby(['Run', 'location']).sum('scaled_score')['scaled_score'].reset_index()

    df_contrib_analysis_ef_op_aop = df_contrib_analysis_ef_op[
        df_contrib_analysis_ef_op.impact_category == ('IMPACT World+ Damage 2.1_regionalized for ecoinvent v3.10', impact_category, f'Total {impact_category.lower()} (biogenic)')
    ].groupby(['Run', 'location']).sum('scaled_score')['scaled_score'].reset_index()

    df_contrib_analysis_ef_res_aop = df_contrib_analysis_ef_res[
        df_contrib_analysis_ef_res.impact_category == ('IMPACT World+ Damage 2.1_regionalized for ecoinvent v3.10', impact_category, f'Total {impact_category.lower()} (biogenic)')
    ].groupby(['Run', 'location']).sum('scaled_score')['scaled_score'].reset_index()

    # Merging the construction, operation and resources dataframes
    df_contrib_analysis_ef_aop = pd.merge(
        left=df_contrib_analysis_ef_constr_aop,
        right=df_contrib_analysis_ef_op_aop,
        on=['Run', 'location'],
        how='outer',
        suffixes=('_constr', '_op'),
    )
    df_contrib_analysis_ef_aop = df_contrib_analysis_ef_aop.merge(
        df_contrib_analysis_ef_res_aop,
        on=['Run', 'location'],
        how='outer',
    ).rename({'scaled_score':'scaled_score_res'}, axis=1).fillna(0)

    # Sum the construction, operation and resources parts
    df_contrib_analysis_ef_aop['scaled_score_tot'] = (
            df_contrib_analysis_ef_aop['scaled_score_constr']
            + df_contrib_analysis_ef_aop['scaled_score_op']
            + df_contrib_analysis_ef_aop['scaled_score_res']
    )

    # Filtering out the 'Not spatialized category'
    # df_contrib_analysis_ef_aop.drop(df_contrib_analysis_ef_aop[df_contrib_analysis_ef_aop.location == 'Not spatialized'].index, inplace=True)

    # Create relative figures with respect to the total score
    df_contrib_analysis_ef_aop['rel_scaled_score_tot'] = df_contrib_analysis_ef_aop.groupby('Run')['scaled_score_tot'].transform(lambda x: x / x.sum())

    df_contrib_analysis_ef_aop = pd.merge(df_contrib_analysis_ef_aop, df_contrib_analysis_ef_aop.groupby('location').max('rel_scaled_score_tot').reset_index()[['location', 'rel_scaled_score_tot']].rename(columns={'rel_scaled_score_tot': 'max_rel_scaled_score_tot'}), on='location', how='left')

In [ ]:
if contribution_ef:
    plot_impact_location(
        df=df_contrib_analysis_ef_aop,
        impact_category=impact_category,
        cutoff=0.01,
        save_results=save_results,
    )

### Climate change, short-term

In [ ]:
df_total_ccst = df_total_impact[['Run', 'index', 'Sector', 'Phase', 'Climate change, short term, total', 'Climate change, short term, total (territorial)', 'Climate change, short term, total (abroad)']].copy(deep=True)

In [ ]:
df_total_ccst = aggregate_mobility_submodels(df_total_ccst)
df_total_ccst['Run'] = df_total_ccst['Run'].replace(reg_level_name_dict)
df_total_ccst['index'] = df_total_ccst['index'].replace(es_tech_name_dict)

In [ ]:
df_ccst = pd.DataFrame(
    results_phases_list_cc,
    columns=['Regionalization level', 'Construction', 'Operation', 'Operation (direct)', 'Resource (wo biomass)', 'Resource (biomass)']
)
df_ccst['Operation (indirect)'] = df_ccst['Operation'] - df_ccst['Operation (direct)']
df_ccst['Total'] = df_ccst[['Construction', 'Operation', 'Resource (wo biomass)', 'Resource (biomass)']].sum(axis=1)
df_ccst.drop(columns=['Operation'], inplace=True)
df_ccst

In [ ]:
(df_ccst['Operation (direct)'] + df_ccst['Resource (biomass)'])

In [ ]:
(df_ccst['Operation (direct)'] + df_ccst['Resource (biomass)']) / df_ccst['Total']

In [ ]:
if save_results:
    df_ccst_copy = df_ccst.copy()
    for col in ['Construction', 'Operation (direct)', 'Operation (indirect)', 'Resource (wo biomass)', 'Resource (biomass)', 'Total']:
        df_ccst_copy[col] *= N_capita * 1e-3  # coming back to original values in [kt CO2-eq]
    df_ccst_copy.to_csv('../03_Results/Tables/reference/ccst.csv', index=False)

In [ ]:
df_ccst_terr = df_total_ccst.groupby('Run').sum()[['Climate change, short term, total', 'Climate change, short term, total (territorial)', 'Climate change, short term, total (abroad)']].rename(columns={'Climate change, short term, total': 'Total CC', 'Climate change, short term, total (territorial)': 'Territorial CC', 'Climate change, short term, total (abroad)': 'Abroad CC'}) * N_capita/1e3
df_ccst_terr = df_ccst_terr.reindex(run_order_2020).reset_index()

In [ ]:
if save_results:
    df_ccst_terr.to_csv('../03_Results/Tables/reference/ccst_terr_abroad.csv', index=False)

In [ ]:
plot_contribution_by_sector(
    df=df_total_ccst,
    imp_cat='Climate change, short term, total',
    save_results=save_results,
    group_by='index',
    cutoff=0.03,
    showlegend=True,
    df_ccst_terr_abroad=df_ccst_terr,
    year=2020,
    show_direct_marker=False,
    show_total_marker=False,
    x_label_name="Regionalization level",
)

In [ ]:
plot_contribution_by_sector(
    df=df_total_ccst,
    imp_cat='Climate change, short term, total',
    save_results=save_results,
    showlegend=True,
    # df_ccst_terr_abroad=df_ccst_terr,
    hatch_phase=False,
    year=2020,
    show_direct_marker=False,
    show_total_marker=True,
    x_label_name="Regionalization level",
)

In [ ]:
plot_contribution_by_sector(
    df=df_total_ccst,
    imp_cat='Climate change, short term, total (territorial)',
    save_results=save_results,
    group_by='index',
    cutoff=0.015,
    showlegend=True,
    df_ccst_terr_abroad=df_ccst_terr,
    year=2020,
    show_direct_marker=False,
    show_total_marker=True,
    x_label_name="Regionalization level",
)

In [ ]:
plot_contribution_by_sector(
    df=df_total_ccst,
    imp_cat='Climate change, short term, total (territorial)',
    save_results=save_results,
    group_by='Sector',
    showlegend=True,
    df_ccst_terr_abroad=df_ccst_terr,
    year=2020,
    show_direct_marker=False,
    show_total_marker=True,
    x_label_name="Regionalization level",
)

In [ ]:
plot_contribution_by_sector(
    df=df_total_ccst,
    imp_cat='Climate change, short term, total (abroad)',
    save_results=save_results,
    group_by='index',
    cutoff=0.01,
    showlegend=True,
    df_ccst_terr_abroad=df_ccst_terr,
    year=2020,
    show_direct_marker=False,
    show_total_marker=True,
    x_label_name="Regionalization level",
)

In [ ]:
plot_contribution_by_sector(
    df=df_total_ccst,
    imp_cat='Climate change, short term, total (abroad)',
    save_results=save_results,
    group_by='Sector',
    showlegend=True,
    df_ccst_terr_abroad=df_ccst_terr,
    year=2020,
    show_direct_marker=False,
    show_total_marker=True,
    x_label_name="Regionalization level",
)

#### Sankey of carbon flows

In [ ]:
for distance_level in ['_SD', '_MD', '_LD', '_ELD']:
    model['Name'] = model['Name'].str.replace(distance_level, '')
    model['Flow'] = model['Flow'].str.replace(distance_level, '')
model.drop_duplicates(inplace=True)

model['Name'] = model['Name'].replace(es_tech_name_dict)

In [ ]:
import plotly.io as pio
pio.renderers.default = "notebook"

In [ ]:
plot_sankey_carbon_flows(
    run='Spat.+Fore.+Back.',
    df_total_impact=df_total_impact,
    cutoff=0,
    model=model,
    aggregate_technologies=True,
    show_figure=True,
    save_results=save_results,
    per_capita=False,
    year=reference_year,
    mode='mfa',
)

### Remaining AoP damages

In [ ]:
list_dfs = []

for reg_level in ['base', 'spat', 'spat_back', 'spat_fore', 'spat_fore_back']:

    # Loading LCA results
    impact_scores = pd.read_csv(path_results+f'2023/{reg_level}/impact_scores.csv')
    impact_scores, impact_abbrev = add_biogenic_climate_change_to_impact_scores_df(impact_scores, impact_abbrev)
    impact_scores, impact_abbrev = add_rhhd_and_reqd_to_impact_scores_df(impact_scores, impact_abbrev)

    # Reading impact categories as tuples
    impact_scores.Impact_category = impact_scores.Impact_category.apply(lambda x: literal_eval(x))

    remaining_aop_impact_categories_list = [
        ('IMPACT World+ Damage 2.1_regionalized for ecoinvent v3.10', 'Ecosystem quality', 'Remaining ecosystem quality'),
        ('IMPACT World+ Damage 2.1_regionalized for ecoinvent v3.10', 'Human health', 'Remaining human health'),
    ]

    # Merging impact scores with energy configuration results
    df_f_mult, df_annual_prod, df_annual_res = get_impact_scores(
        impact_category=remaining_aop_impact_categories_list,
        df_impact_scores=impact_scores,
        df_results=results,
    )

    remaining_hh_constr = df_f_mult['Remaining human health'].sum()
    remaining_hh_op = df_annual_prod['Remaining human health'].sum()
    remaining_hh_res = df_annual_res['Remaining human health'].sum()
    remaining_hh_tot = remaining_hh_constr + remaining_hh_op + remaining_hh_res

    remaining_eq_constr = df_f_mult['Remaining ecosystem quality'].sum()
    remaining_eq_op = df_annual_prod['Remaining ecosystem quality'].sum()
    remaining_eq_res = df_annual_res['Remaining ecosystem quality'].sum()
    remaining_eq_tot = remaining_eq_constr + remaining_eq_op + remaining_eq_res

    df = pd.DataFrame(
        data=[
            [reg_level, 'Remaining human health', remaining_hh_constr, remaining_hh_op, remaining_hh_res, remaining_hh_tot],
            [reg_level, 'Remaining ecosystem quality', remaining_eq_constr, remaining_eq_op, remaining_eq_res, remaining_eq_tot],
        ],
        columns=['Run', 'Impact category', 'Construction', 'Operation', 'Resource', 'Total'],
    )

    list_dfs.append(df)

df_remaining_aop = pd.concat(list_dfs, ignore_index=True)
if save_results:
    df_remaining_aop.to_csv('../03_Results/Tables/reference/remaining_aop.csv', index=False)

In [ ]:
df_remaining_aop

## Assessing abroad emissions adjustment factor for 2050 runs

In [ ]:
impact_categories_list = [
    ('IMPACT World+ Damage 2.1_regionalized for ecoinvent v3.10', 'Ecosystem quality', 'Total ecosystem quality (biogenic)'),
    ('IMPACT World+ Damage 2.1_regionalized for ecoinvent v3.10', 'Human health', 'Total human health (biogenic)'),
    ('IMPACT World+ Damage 2.1_regionalized for ecoinvent v3.10', 'Ecosystem quality', 'Remaining ecosystem quality'),
    ('IMPACT World+ Damage 2.1_regionalized for ecoinvent v3.10', 'Human health', 'Remaining human health'),
    ('IMPACT World+ Midpoint 2.1 for ecoinvent v3.10 (incl. CO2 uptake)', 'Midpoint', 'Climate change, short term, total'),
]

In [ ]:
comparison_2023_2050 = []

for reg_level in ['base', 'spat', 'spat_back', 'spat_fore', 'spat_fore_back']:

    for level in [f'2023/{reg_level}', f'2050/{reg_level}/SSP5-H', f'2050/{reg_level}/SSP2-L']:

        # Loading LCA results
        impact_scores = pd.read_csv(path_results+f'{level}/impact_scores.csv')
        impact_scores_direct = pd.read_csv(path_results+f'{level}/impact_scores_direct_emissions.csv')

        impact_scores = add_biogenic_climate_change_to_impact_scores_df(impact_scores, impact_abbrev)[0]
        impact_scores_direct = add_biogenic_climate_change_to_impact_scores_df(impact_scores_direct, impact_abbrev)[0]

        impact_scores = add_rhhd_and_reqd_to_impact_scores_df(impact_scores, impact_abbrev)[0]
        impact_scores_direct = add_rhhd_and_reqd_to_impact_scores_df(impact_scores_direct, impact_abbrev)[0]

        # from [kg CO2-eq / kW(h) or pkm(/h) or tkm(/h)] to [t-CO2-eq / GW(h) or Mpkm(/h) or Mtkm(/h)]
        impact_scores.Value *= 1e3
        impact_scores_direct.Value *= 1e3

        # Reading impact categories as tuples
        impact_scores.Impact_category = impact_scores.Impact_category.apply(lambda x: literal_eval(x))
        impact_scores_direct.Impact_category = impact_scores_direct.Impact_category.apply(lambda x: literal_eval(x))

        # Merging impact scores with energy configuration results
        df_f_mult, df_annual_prod, df_annual_res = get_impact_scores(
            impact_category=impact_categories_list,
            df_impact_scores=impact_scores,
            df_results=results,
        )

        df_annual_prod_direct = get_impact_scores(
            impact_category=impact_categories_list,
            df_impact_scores=impact_scores_direct,
            df_results=results,
            assessment_type='direct',
        )

        for cat in impact_categories_list:

            impact_constr = df_f_mult[cat[-1]].sum()
            impact_op = df_annual_prod[cat[-1]].sum()
            impact_op_direct = df_annual_prod_direct[cat[-1]].sum()
            impact_res_wo_biomass = df_annual_res[~df_annual_res['index'].isin(wood_list+wet_biomass_list+waste_list)][cat[-1]].sum()
            impact_res_biomass = df_annual_res[df_annual_res['index'].isin(wood_list+wet_biomass_list+waste_list)][cat[-1]].sum()

            impact_tot = impact_constr + impact_op + impact_res_wo_biomass + impact_res_biomass
            impact_op_indirect = impact_op - impact_op_direct

            comparison_2023_2050.append([
                cat[-1],
                level,
                impact_constr,
                impact_op_direct,
                impact_op_indirect,
                impact_res_wo_biomass,
                impact_res_biomass,
                impact_tot,
            ])

comparison_2023_2050 = pd.DataFrame(comparison_2023_2050, columns=['Impact category', 'Database', 'Construction', 'Operation (direct)', 'Operation (indirect)', 'Resources (wo biomass)', 'Resources (biomass)', 'Total']).set_index(['Impact category', 'Database'])

In [ ]:
comparison_2023_2050.head()

In [ ]:
adjustment_ratios = []

for ssp_rcp in ['SSP5-H', 'SSP2-L']:
    for reg_level in ['base', 'spat', 'spat_back', 'spat_fore', 'spat_fore_back']:
        res_2050 = comparison_2023_2050.loc['Climate change, short term, total'].loc[f'2050/{reg_level}/{ssp_rcp}']
        abroad_2050 = res_2050['Total'] - (res_2050['Operation (direct)'] + res_2050['Resources (biomass)'])
        res_2023 = comparison_2023_2050.loc['Climate change, short term, total'].loc[f'2023/{reg_level}']
        abroad_2023 = res_2023['Total'] - (res_2023['Operation (direct)'] + res_2050['Resources (biomass)'])
        adjustment_ratios.append(['Climate change, short term, total', 'Abroad', reg_level, ssp_rcp, abroad_2050 / abroad_2023])

for cat in ['Remaining human health', 'Remaining ecosystem quality']:
    for ssp_rcp in ['SSP5-H', 'SSP2-L']:
        for reg_level in ['base', 'spat', 'spat_back', 'spat_fore', 'spat_fore_back']:
            tot_2050 = comparison_2023_2050.loc[cat].loc[f'2050/{reg_level}/{ssp_rcp}']['Total']
            tot_2023 = comparison_2023_2050.loc[cat].loc[f'2023/{reg_level}']['Total']
            adjustment_ratios.append([cat, 'Total', reg_level, ssp_rcp, tot_2050 / tot_2023])

In [ ]:
df_adjustment_ratios = pd.DataFrame(adjustment_ratios, columns=['Impact category', 'Scope', 'Regionalization level', 'RCP', 'Ratio'])
df_adjustment_ratios.head()

In [ ]:
if save_results:
    df_adjustment_ratios.to_csv('../03_Results/Tables/reference/adjustment_ratios.csv', index=False)